# 0. Problem
## 1978. Employees Whose Manager Left the Company — Easy
Return employee IDs with salary < 30000 whose manager ID is non-null but no longer exists in the employee table.
Official: https://leetcode.com/problems/employees-whose-manager-left-the-company/

# 1. Setup

In [ ]:
import pandas as pd
employees_rows=[(3,"Mila",9,60301),(12,"Anton",None,31000),(13,"Emery",None,67084),(1,"Kalel",11,21241),(9,"Mikaela",None,50937),(11,"Joziah",6,28485)]
employees_pd=pd.DataFrame(employees_rows,columns=["employee_id","name","manager_id","salary"]); employees_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark=SparkSession.builder.getOrCreate(); employees_spark=spark.createDataFrame(employees_rows,"employee_id int, name string, manager_id int, salary int"); employees_spark.createOrReplaceTempView("Employees")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""SELECT e.employee_id FROM Employees e WHERE e.salary<30000 AND e.manager_id IS NOT NULL AND NOT EXISTS (SELECT 1 FROM Employees m WHERE m.employee_id=e.manager_id) ORDER BY e.employee_id"""); sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
existing=set(employees_pd["employee_id"]); result_pd=employees_pd.loc[employees_pd["salary"].lt(30000)&employees_pd["manager_id"].notna()&~employees_pd["manager_id"].isin(existing),["employee_id"]].sort_values("employee_id").reset_index(drop=True); result_pd

# 4. PySpark Solution

In [ ]:
candidates=employees_spark.filter((F.col("salary")<30000)&F.col("manager_id").isNotNull()); managers=employees_spark.select(F.col("employee_id").alias("manager_id")).distinct(); result_spark=candidates.join(managers,on="manager_id",how="left_anti").select("employee_id").orderBy("employee_id"); result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| missing referenced row | `NOT EXISTS` | `~.isin(set)` | `left_anti` |
| non-null FK | `IS NOT NULL` | `.notna()` | `.isNotNull()` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Employees

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: employees_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: employees_spark